In [35]:
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import json

print("Loading data...")
df = pd.read_csv(r"c:\Users\loq\Desktop\Trading\finalgo\data\ranking_data_upstox_30min_1y.csv")

print("Loading models and metadata...")
with open(r"c:\Users\loq\Desktop\Trading\finalgo\models\v1_30min\metadata.json", 'r') as f:
    metadata = json.load(f)
features = metadata['features']
print(f"Num features: {len(features)}")

long_model = xgb.Booster()
long_model.load_model(r"c:\Users\loq\Desktop\Trading\finalgo\models\v1_30min\xgb_long_model.json")
short_model = xgb.Booster()
short_model.load_model(r"c:\Users\loq\Desktop\Trading\finalgo\models\v1_30min\xgb_short_model.json")

long_model.feature_names = features
short_model.feature_names = features

df = df.dropna(subset=features + ['Next_30Min_Return'])
print(f"Data shape after NA drop: {df.shape}")

Loading data...


Loading models and metadata...
Num features: 95
Data shape after NA drop: (541143, 105)


In [34]:
target_cols = [c for c in df.columns if 'Return' in c or 'Target' in c or 'Next' in c]
print("Potential Target Columns:", target_cols)
print("First few columns:", df.columns[:10].tolist())
print("Last few columns:", df.columns[-10:].tolist())

Potential Target Columns: ['Return', 'Log_Return', 'Return_lag1', 'Return_lag2', 'Return_lag3', 'Intraday_Return', 'Return_Accel', 'Next_30Min_Return', 'Market_Mean_Return', 'Relative_Return']
First few columns: ['Open', 'High', 'Low', 'Close', 'Volume', 'DateTime', 'Ticker', 'Return', 'Log_Return', 'HL_Range']
Last few columns: ['Raw_Volume_Zscore', 'CMF_8', 'RVOL_4', 'Next_30Min_Return', 'DateTime_30Min', 'Query_ID', 'Market_Mean_Return', 'Relative_Return', 'Market_Mean_Volatility', 'Relative_Volatility']


In [36]:
# Predict
X = df[features]
dmatrix = xgb.DMatrix(X)
df['Long_Pred'] = long_model.predict(dmatrix)
df['Short_Pred'] = short_model.predict(dmatrix)

# 1. Time-of-Day Analysis
if 'DateTime' in df.columns:
    df['Datetime_parsed'] = pd.to_datetime(df['DateTime'])
    df['Time'] = df['Datetime_parsed'].dt.strftime('%H:%M')

print("1. Time-of-Day Analysis")
# Long Edge (Top-5% confidence scores)
long_threshold = df['Long_Pred'].quantile(0.95)
long_edge = df[df['Long_Pred'] >= long_threshold].groupby('Time')['Next_30Min_Return'].agg(['mean', 'count'])
print(f"\nLong Edge (Score >= {long_threshold:.4f}) by Time:")
print(long_edge)

short_threshold = df['Short_Pred'].quantile(0.95)
# Note: For short, a negative return is a win. We'll show the actual raw mean return.
short_edge = df[df['Short_Pred'] >= short_threshold].groupby('Time')['Next_30Min_Return'].agg(['mean', 'count'])
print(f"\nShort Edge (Score >= {short_threshold:.4f}) by Time:")
print(short_edge)

# 2. Volatility Analysis
print("\n2. Volatility Analysis")
vol_col = 'Market_Mean_Volatility' if 'Market_Mean_Volatility' in df.columns else 'HL_Range'
df['Vol_Bucket'] = pd.qcut(df[vol_col], q=4, labels=['Low', 'Med-Low', 'Med-High', 'High'])

long_vol = df[df['Long_Pred'] >= long_threshold].groupby('Vol_Bucket')['Next_30Min_Return'].mean()
print("\nLong Edge (Top 5%) Returns by Volatility Regime:")
print(long_vol)

short_vol = df[df['Short_Pred'] >= short_threshold].groupby('Vol_Bucket')['Next_30Min_Return'].mean()
print("\nShort Edge (Top 5%) Returns by Volatility Regime:")
print(short_vol)

# 3. Score Thresholding Analysis
print("\n3. Score Thresholding Analysis")
# Group all data into deciles of Long_Pred
df['Long_Decile'] = pd.qcut(df['Long_Pred'], q=10, labels=False)
long_decile_returns = df.groupby('Long_Decile')['Next_30Min_Return'].mean()
print("\nLong Returns by Score Decile (0=Lowest, 9=Highest):")
print(long_decile_returns)

df['Short_Decile'] = pd.qcut(df['Short_Pred'], q=10, labels=False)
short_decile_returns = df.groupby('Short_Decile')['Next_30Min_Return'].mean()
print("\nShort Returns by Score Decile (0=Lowest, 9=Highest):")
print(short_decile_returns)

with open('30m_edge_results.txt', 'w') as f:
    f.write(f"Long Edge by Time:\n{long_edge}\n\nShort Edge by Time:\n{short_edge}\n\nLong Vol:\n{long_vol}\n\nShort Vol:\n{short_vol}\n\nLong Decile:\n{long_decile_returns}\n\nShort Decile:\n{short_decile_returns}")

1. Time-of-Day Analysis

Long Edge (Score >= 0.0417) by Time:
           mean  count
Time                  
09:15  0.000855   1378
09:45  0.000991   1044
10:15  0.000603   1159
10:45  0.000871   1237
11:15  0.000772   1274
11:45  0.000782   1436
12:15  0.000673   1502
12:45  0.000410   1543
13:15  0.000687   1929
13:45  0.000756   2065
14:15  0.000715   2906
14:45  0.000624   3864
15:15  0.002379   5721

Short Edge (Score >= 0.0382) by Time:
           mean  count
Time                  
09:15 -0.001089   1644
09:45 -0.000688   1127
10:15 -0.000854   1173
10:45 -0.001023   1045
11:15 -0.000675   1232
11:45 -0.000848   1188
12:15 -0.000548   1235
12:45 -0.001016   1246
13:15 -0.000629   1537
13:45 -0.000478   1585
14:15 -0.000584   4192
14:45 -0.001213   4390
15:15 -0.002268   5464

2. Volatility Analysis

Long Edge (Top 5%) Returns by Volatility Regime:
Vol_Bucket
Low         0.000919
Med-Low     0.000952
Med-High    0.000835
High        0.001661
Name: Next_30Min_Return, dtype: float64



Short Edge (Top 5%) Returns by Volatility Regime:
Vol_Bucket
Low        -0.000656
Med-Low    -0.001056
Med-High   -0.001549
High       -0.001260
Name: Next_30Min_Return, dtype: float64

3. Score Thresholding Analysis

Long Returns by Score Decile (0=Lowest, 9=Highest):
Long_Decile
0   -0.000536
1    0.000049
2   -0.000078
3   -0.000045
4   -0.000019
5    0.000050
6    0.000073
7   -0.000047
8    0.000041
9    0.000715
Name: Next_30Min_Return, dtype: float64

Short Returns by Score Decile (0=Lowest, 9=Highest):
Short_Decile
0    0.000486
1    0.000134
2    0.000067
3    0.000031
4   -0.000008
5   -0.000007
6    0.000033
7    0.000150
8    0.000052
9   -0.000735
Name: Next_30Min_Return, dtype: float64
